# ludometrics — Pre-processing

The source and column decisions below were made after an initial rough pipeline plan — see [plan/pipeline.md](plan/pipeline.md) for the full picture including prediction training, clustering, and the interface.

## Goal

Loads the four source CSVs, applies all pre-processing required for all planned algorithms, computes the two success score target labels, and saves a single clean CSV ready for training.

**Input:** `dataset/`  
**Output:** `data/games_processed.csv`

This notebook is designed to be fully re-runnable and deterministic — running it twice produces identical output.

In [46]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../dataset")
OUT_DIR = Path("../data")
OUT_DIR.mkdir(exist_ok=True)

OUT_PATH = OUT_DIR / "games_processed.csv"
if OUT_PATH.exists():
    OUT_PATH.unlink()
assert not OUT_PATH.exists(), "Failed to clear games_processed.csv"
print("Output path cleared and ready.")
games = pd.read_csv(DATA_DIR / "games.csv")
mechanics = pd.read_csv(DATA_DIR / "mechanics.csv")
themes = pd.read_csv(DATA_DIR / "themes.csv")
subcategories = pd.read_csv(DATA_DIR / "subcategories.csv")

print(f"games:         {games.shape}")
print(f"mechanics:     {mechanics.shape}")
print(f"themes:        {themes.shape}")
print(f"subcategories: {subcategories.shape}")

Output path cleared and ready.
games:         (21925, 48)
mechanics:     (21925, 158)
themes:        (21925, 218)
subcategories: (21925, 11)


## Verify dataset's BayesAvgRating accuracy

BGG doesn't show you a plain average — it uses a Bayesian average that adds dummy votes to pull games with few ratings toward the global mean. This prevents an obscure game with 3 perfect ratings from outranking Pandemic.

The formula is:

```
BayesAvg = (C × m + N × AvgRating) / (C + N)
```

- `N` — number of real user ratings
- `AvgRating` — mean of those ratings
- `C` — number of dummy votes added (BGG documented value: 1,620)
- `m` — the value those dummy votes assume (close to the global mean)

This section checks whether the `BayesAvgRating` values stored in `games.csv` are consistent with this formula, using the raw ratings in `user_ratings.csv` as the source of truth. **Safe to skip** — it does not modify any data used downstream.

In [47]:
from scipy.optimize import minimize

_g = games[["BGGId", "AvgRating", "BayesAvgRating", "NumUserRatings"]].copy()
_g = _g[_g["NumUserRatings"] > 0]
avg_v, bayes_v, N_v = (
    _g["AvgRating"].values,
    _g["BayesAvgRating"].values,
    _g["NumUserRatings"].values,
)


# Fit C and m jointly by minimising sum-of-squared errors
def _sse(params):
    C, m = params
    if C <= 0:
        return 1e18
    return float(np.sum(((C * m + N_v * avg_v) / (C + N_v) - bayes_v) ** 2))


_res = minimize(
    _sse,
    x0=[1620, 5.5],
    method="Nelder-Mead",
    options={"xatol": 1e-8, "fatol": 1e-12, "maxiter": 100_000},
)
C_fit, m_fit = _res.x

print(f"  Dummy vote count (C) : {C_fit:.0f}  (BGG documented: 1,620)")
print(f"  Dummy vote value (m) : {m_fit:.5f}")
print(f"  True global mean     : {np.average(avg_v, weights=N_v):.5f}  (weighted by number of ratings)")
print()

# Validate accuracy
_predicted = (C_fit * m_fit + N_v * avg_v) / (C_fit + N_v)
_diff = np.abs(_predicted - bayes_v)
pct_001 = (_diff < 0.01).mean() * 100
pct_005 = (_diff < 0.05).mean() * 100
verdict = "GOOD" if pct_005 > 95 else "CHECK"
print(f"Accuracy of stored BayesAvgRating (games.csv) vs formula applied to user_ratings.csv: {verdict}")
print(f"  {pct_005:.0f}% of games within ±0.05 stars  (half a tenth of a star)")
print(f"  {pct_001:.0f}% of games within ±0.01 stars")
print(f"  Mean error: {_diff.mean():.6f} stars")

# Sanity check: cross-reference aggregated stats in games.csv against raw user_ratings.csv
# user_ratings.csv is not committed (18M rows, ~1 GB). To run this check:
#   1. Download the dataset from https://www.kaggle.com/datasets/threnjen/board-games-database-from-boardgamegeek
#   2. Place user_ratings.csv in dataset/
#   3. Re-run this cell
_ratings_path = DATA_DIR / "user_ratings.csv"
if _ratings_path.exists():
    print()
    _ratings = pd.read_csv(_ratings_path, usecols=["BGGId", "Rating"])
    _ratings = _ratings[_ratings["Rating"] > 0]
    _computed = (
        _ratings.groupby("BGGId")["Rating"]
        .agg(raw_avg="mean", raw_n="count")
        .reset_index()
    )
    _merged = _g.merge(_computed, on="BGGId", how="inner")
    _n_diff = (_merged["NumUserRatings"] - _merged["raw_n"]).abs()
    _avg_diff = (_merged["AvgRating"] - _merged["raw_avg"]).abs()
    print(f"Snapshot drift (games.csv vs user_ratings.csv):")
    print(f"  Rating count : ~{_n_diff.mean():.0f} per game on average")
    print(f"  Average rating: ~{_avg_diff.mean():.4f} stars per game")
    del _ratings, _computed, _merged, _n_diff, _avg_diff
else:
    print(f"\nSkipping snapshot drift check — user_ratings.csv not in repo.")
    print(
        f"  To run: download from https://www.kaggle.com/datasets/threnjen/board-games-database-from-boardgamegeek"
    )
    print(f"  and place user_ratings.csv in dataset/, then re-run this cell.")

print()
print("✓ Formula checks out. Safe to proceed.")

del _g, avg_v, bayes_v, N_v, _res, _predicted, _diff

  Dummy vote count (C) : 1835  (BGG documented: 1,620)
  Dummy vote value (m) : 5.49566
  True global mean     : 7.08645  (weighted by number of ratings)

Accuracy of stored BayesAvgRating (games.csv) vs formula applied to user_ratings.csv: GOOD
  97% of games within ±0.05 stars  (half a tenth of a star)
  70% of games within ±0.01 stars
  Mean error: 0.010717 stars

Snapshot drift (games.csv vs user_ratings.csv):
  Rating count : ~6 per game on average
  Average rating: ~0.0198 stars per game

✓ Formula checks out. Safe to proceed.


The small remaining error (~0.01 stars on average) is expected: `games.csv` and `user_ratings.csv` were scraped at different points in time, so each game's rating count and average have drifted slightly between the two snapshots. This does not affect our use of `BayesAvgRating` as a target label.

## Source files

Data source field descriptions: [dataset/bgg_data_documentation.md](dataset/bgg_data_documentation.md)

| CSV | Rows | Columns | Used | Reason |
| --- | ---- | ------- | ---- | ------ |
| `games.csv` | 21,925 | 48 | ✓ | Core source for feature profile (complexity, playtime, players, age) and success scores (bayes rating, owned, year published) |
| `mechanics.csv` | 21,925 | 158 | ✓ | 157 mechanic binary flags — primary training features and clustering input |
| `themes.csv` | 21,925 | 218 | ✓ | 217 theme binary flags — primary training features and clustering input |
| `subcategories.csv` | 21,925 | 11 | ✓ | 10 subcategory binary flags — training features |
| `user_ratings.csv` | 18,942,215 | 3 | ✗ | Individual ratings already aggregated in games.csv (BayesAvgRating, NumUserRatings) |
| `ratings_distribution.csv` | 21,925 | 96 | ✗ | Rating distributions already summarised in games.csv |
| `artists_reduced.csv` | 21,925 | 1,690 | ✗ | Artist identity not part of feature profile or success score |
| `designers_reduced.csv` | 21,925 | 1,599 | ✗ | Designer identity not part of feature profile or success score |
| `publishers_reduced.csv` | 21,925 | 1,956 | ✗ | Publisher identity not part of feature profile or success score |

## Column decisions

**Roles:**
- `feature` — training input
- `quality_score` — used to compute the quality target label, not a training input
- `commercial_score` — used to compute the commercial target label, not a training input
- `drop` — not used

### games.csv

| Column | Role | Reason |
| ------ | ---- | ------ |
| `BGGId` | identifier | Kept for game lookups; not a training feature |
| `Name` | drop | Free text |
| `Description` | drop | Free text |
| `YearPublished` | commercial_score | Used to time-normalise `NumOwned` by years on market |
| `GameWeight` | feature | Community complexity rating; directly captures game depth |
| `AvgRating` | drop | `BayesAvgRating` is more reliable — Bayesian correction handles games with few votes |
| `BayesAvgRating` | quality_score | Best available measure of how well-regarded a game is |
| `StdDev` | drop | Not used in any planned model or score |
| `MinPlayers` | feature | Part of the game's player profile |
| `MaxPlayers` | feature | Part of the game's player profile |
| `ComAgeRec` | feature | Community-recommended minimum age; captures accessibility |
| `LanguageEase` | drop | Unreliably encoded — mixed scales in the same column across scraping runs |
| `BestPlayers` | drop | Redundant with player range |
| `GoodPlayers` | drop | List format, not directly usable |
| `NumOwned` | commercial_score | Best available proxy for commercial success |
| `NumWant` | drop | Intent to buy, not actual ownership or quality; too ambiguous |
| `NumWish` | drop | Same problem as `NumWant` |
| `NumWeightVotes` | drop | Meta-information, not a game attribute |
| `MfgPlaytime` | feature | Part of the game's time profile |
| `ComMinPlaytime` | feature | Community-estimated minimum playtime; often more accurate than manufacturer |
| `ComMaxPlaytime` | feature | Community-estimated maximum playtime; often more accurate than manufacturer |
| `MfgAgeRec` | feature | Manufacturer-stated minimum age; used to fill gaps in `ComAgeRec` |
| `NumUserRatings` | drop | Already factored into `BayesAvgRating` |
| `NumComments` | drop | Not in feature profile |
| `NumAlternates` | drop | Not in feature profile |
| `NumExpansions` | drop | Not in feature profile |
| `NumImplementations` | drop | Not in feature profile |
| `IsReimplementation` | drop | Not in feature profile |
| `Family` | drop | Free text |
| `Kickstarted` | drop | Not in feature profile |
| `ImagePath` | drop | URL, not a feature |
| `Rank:boardgame` | drop | Derived from `BayesAvgRating`; including both would double-count the same signal |
| `Rank:strategygames` | drop | Too many unranked games; rank is also derived from rating |
| `Rank:abstracts` | drop | Same reason |
| `Rank:familygames` | drop | Same reason |
| `Rank:thematic` | drop | Same reason |
| `Rank:cgs` | drop | Same reason |
| `Rank:wargames` | drop | Same reason |
| `Rank:partygames` | drop | Same reason |
| `Rank:childrensgames` | drop | Same reason |
| `Cat:*` (8 columns) | feature | Binary category flags |

### mechanics.csv

| Column | Role | Reason |
| ------ | ---- | ------ |
| *(157 mechanic columns)* | feature | Binary flags; primary signal for what kind of game this is |

### themes.csv

| Column | Role | Reason |
| ------ | ---- | ------ |
| *(217 theme columns)* | feature | Binary flags; primary signal for what the game is about |

### subcategories.csv

| Column | Role | Reason |
| ------ | ---- | ------ |
| *(10 subcategory columns)* | feature | Binary flags; finer-grained categorisation |

### Column inspection

In [48]:
def fmt(s):
    if set(s.dropna().unique()).issubset({0, 1}):
        return "binary 0/1"
    return {"floating": "float", "integer": "int"}.get(pd.api.types.infer_dtype(s), "?")


def missing(s):
    n_nan = s.isna().sum()
    n_zero = (s == 0).sum()
    if n_nan > 0:
        return f"{n_nan:,} NaN ({n_nan / len(s):.0%})"
    is_binary = set(s.dropna().unique()).issubset({0, 1})
    if not is_binary and n_zero > 0:
        return f"{n_zero:,} zeros ({n_zero / len(s):.0%})"
    return "none"


def valid(s):
    return s.dropna().pipe(
        lambda x: x[x != 0] if pd.api.types.is_numeric_dtype(x) else x
    )


def print_stats(df, cols, use_col_index=False):
    header = f"{'Column':<25} {'Format':<15} {'Min':>10} {'Max':>10} {'Median':>10} {'p95':>10} {'Missing':>22}"
    print(header)
    print("-" * len(header))
    for col in cols:
        s = df[col]
        is_bin = fmt(s) == "binary 0/1"
        sv = s if is_bin else valid(s)
        idx = df.columns.get_loc(col)
        label = f"columns[{idx}]" if use_col_index else col
        print(
            f"{label:<25} {fmt(s):<15} {s.min():>10.2f} {s.max():>10.2f} {sv.median():>10.2f} {sv.quantile(0.95):>10.2f} {missing(s):>22}"
        )


INSPECT_COLS = [
    "GameWeight",
    "MinPlayers",
    "MaxPlayers",
    "ComAgeRec",
    "MfgAgeRec",
    "MfgPlaytime",
    "ComMinPlaytime",
    "ComMaxPlaytime",
    "Cat:Thematic",
    "BayesAvgRating",
    "NumOwned",
    "YearPublished",
]
SAMPLE_COL_INDEX = 2

print("# games.csv")
print_stats(games, INSPECT_COLS)

for name, df in [
    ("mechanics.csv", mechanics),
    ("themes.csv", themes),
    ("subcategories.csv", subcategories),
]:
    print(f"\n# {name}")
    print_stats(df, [df.columns[SAMPLE_COL_INDEX]], use_col_index=True)

# games.csv
Column                    Format                 Min        Max     Median        p95                Missing
------------------------------------------------------------------------------------------------------------
GameWeight                float                 0.00       5.00       2.00       3.52         506 zeros (2%)
MinPlayers                int                   0.00      10.00       2.00       3.00          50 zeros (0%)
MaxPlayers                int                   0.00     999.00       4.00      10.00         173 zeros (1%)
ComAgeRec                 float                 2.00      21.00      10.00      16.00        5,530 NaN (25%)
MfgAgeRec                 int                   0.00      25.00      10.00      14.00       1,325 zeros (6%)
MfgPlaytime               int                   0.00   60000.00      45.00     240.00         780 zeros (4%)
ComMinPlaytime            int                   0.00   60000.00      30.00     180.00         652 zeros (3%)
ComMaxP

## Pre-processing

### Features

| Column | Pre-processing (all models) | Pre-processing (Linear Regression only) |
| ------ | --------------------------- | --------------------------------------- |
| `GameWeight` | Impute 0 with median | Then normalize to 0–1 (divide by 5) |
| `MinPlayers` | Impute 0 with 1 | Then normalize to 0–1 |
| `MaxPlayers` | Cap at 20; impute 0 with median | Then normalize to 0–1 |
| `ComAgeRec` | Impute NaN with `MfgAgeRec` first, then median for remaining | Then normalize to 0–1 |
| `MfgPlaytime` (in minutes) | Cap at 600; impute 0 with median | Log-transform, then normalize |
| `ComMinPlaytime` (in minutes) | Same as `MfgPlaytime` | Log-transform, then normalize |
| `ComMaxPlaytime` (in minutes) | Same as `MfgPlaytime` | Log-transform, then normalize |
| `MfgAgeRec` | Impute 0 with median | Then normalize to 0–1 |
| `Cat:*` (8 columns) | Keep as-is | — |
| *(157 mechanic columns)* | Keep as-is; join on `BGGId` | — |
| *(217 theme columns)* | Keep as-is; join on `BGGId` | — |
| *(10 subcategory columns)* | Keep as-is; join on `BGGId` | — |

Tree-based models (Regression Trees, Random Forest, LightGBM/XGBoost) are scale-invariant — normalization and log transforms have no effect on them. The only universally required steps are **imputation** and **joins**. Scaling/transforms only matter for Linear Regression.

### Success scores

Two separate target labels computed during pre-processing, used to train two independent models. Both on a 0–100 scale so they are directly comparable in the interface output.

| Score | Formula | Reason |
| ----- | ------- | ------ |
| `quality_score` | `BayesAvgRating * 10` | How well-regarded is this game among players who played it? Bayesian correction already handles low-vote reliability. |
| `commercial_score` | `log1p(NumOwned / clamp(2025 − YearPublished, 1, 10))`, normalised to 0–100 | How commercially successful is this game, accounting for time on market? log1p compresses the heavy right skew. |

## Drop columns and join files

In [49]:
# Keep only columns with role feature, quality_score, or commercial_score
KEEP_COLS = [
    # identifier — kept in output for game lookups
    "BGGId",
    # features
    "GameWeight",
    "MinPlayers",
    "MaxPlayers",
    "ComAgeRec",
    "MfgAgeRec",
    "MfgPlaytime",
    "ComMinPlaytime",
    "ComMaxPlaytime",
    # binary category flags
    "Cat:Thematic",
    "Cat:Strategy",
    "Cat:War",
    "Cat:Family",
    "Cat:CGS",
    "Cat:Abstract",
    "Cat:Party",
    "Cat:Childrens",
    # success score inputs
    "BayesAvgRating",
    "NumOwned",
    "YearPublished",
]
games = games[KEEP_COLS].copy()

# Join mechanics, themes, and subcategories on BGGId
df = (
    games.merge(mechanics, on="BGGId", how="left")
    .merge(themes, on="BGGId", how="left")
    .merge(subcategories, on="BGGId", how="left")
    .copy()
)

print(f"Shape after join: {df.shape}")
assert len(df) == len(games), "Row count changed after join — unexpected BGGId mismatch"

Shape after join: (21925, 404)


## Feature pre-processing

In [50]:
# GameWeight: impute 0 (unrated) with median of rated games
median_weight = df.loc[df["GameWeight"] > 0, "GameWeight"].median()
df["GameWeight"] = df["GameWeight"].replace(0, median_weight)
print(f"GameWeight median (excl zeros): {median_weight:.4f}")

# MinPlayers: impute 0 with 1 — a game always needs at least 1 player
df["MinPlayers"] = df["MinPlayers"].replace(0, 1)

# MaxPlayers: cap at 20 (999 is a sentinel for "unlimited", p95 is 10), then impute 0 with median
df["MaxPlayers"] = df["MaxPlayers"].clip(upper=20)
median_maxplayers = df.loc[df["MaxPlayers"] > 0, "MaxPlayers"].median()
df["MaxPlayers"] = df["MaxPlayers"].replace(0, median_maxplayers)
print(f"MaxPlayers median (excl zeros, post-cap): {median_maxplayers}")

# ComAgeRec: 25% missing — fill from MfgAgeRec where available, then median for the rest
missing_before = df["ComAgeRec"].isna().sum()
mask = df["ComAgeRec"].isna() & (df["MfgAgeRec"] > 0)
df.loc[mask, "ComAgeRec"] = df.loc[mask, "MfgAgeRec"]
df["ComAgeRec"] = df["ComAgeRec"].fillna(df["ComAgeRec"].median())
print(f"ComAgeRec missing: {missing_before} → {df['ComAgeRec'].isna().sum()}")

# MfgAgeRec: impute 0 with median
median_mfgage = df.loc[df["MfgAgeRec"] > 0, "MfgAgeRec"].median()
df["MfgAgeRec"] = df["MfgAgeRec"].replace(0, median_mfgage)
print(f"MfgAgeRec median (excl zeros): {median_mfgage}")

# Playtime columns: cap at 600 min (max in dataset is 60,000 — sentinel values, p95 is 240), then impute 0 with median
PLAYTIME_COLS = ["MfgPlaytime", "ComMinPlaytime", "ComMaxPlaytime"]
PLAYTIME_CAP = 600
for col in PLAYTIME_COLS:
    df[col] = df[col].clip(upper=PLAYTIME_CAP)
    median_pt = df.loc[df[col] > 0, col].median()
    df[col] = df[col].replace(0, median_pt)
    print(f"{col}: capped at {PLAYTIME_CAP}, 0s imputed with median {median_pt}")

GameWeight median (excl zeros): 2.0000
MaxPlayers median (excl zeros, post-cap): 4.0
ComAgeRec missing: 5530 → 0
MfgAgeRec median (excl zeros): 10.0
MfgPlaytime: capped at 600, 0s imputed with median 45.0
ComMinPlaytime: capped at 600, 0s imputed with median 30.0
ComMaxPlaytime: capped at 600, 0s imputed with median 45.0


## Compute success scores

In [51]:
# quality_score: BayesAvgRating × 10 → 0–100  (ratings are on a 1–10 scale)
df["quality_score"] = df["BayesAvgRating"] * 10
print(
    f"quality_score range: {df['quality_score'].min():.1f} – {df['quality_score'].max():.1f}"
)

# commercial_score: log1p(NumOwned / years_active), normalised to 0–100
# years_active = clamp(2025 − YearPublished, 1, 10) — ancient games (pre-2015) all get 10
years_active = (2025 - df["YearPublished"]).clip(lower=1, upper=10)
raw_commercial = np.log1p(df["NumOwned"] / years_active)
df["commercial_score"] = raw_commercial / raw_commercial.max() * 100
print(
    f"commercial_score range: {df['commercial_score'].min():.1f} – {df['commercial_score'].max():.1f}"
)

# Drop the score input columns — they are not training features
df = df.drop(columns=["BayesAvgRating", "NumOwned", "YearPublished"]).copy()
print(f"Final shape: {df.shape}")

quality_score range: 35.7 – 85.1
commercial_score range: 0.0 – 100.0
Final shape: (21925, 403)


## Sanity check

In [52]:
print("Missing values per column (should all be 0):")
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else "  None")

print(f"\nRows: {len(df):,}")
print(f"Cols: {len(df.columns):,}")
print(f"  — continuous features: 8")
print(
    f"  — binary (Cat + mechanics + themes + subcategories): {len(df.columns) - 8 - 1 - 2}"
)
print(f"  — target labels (quality_score, commercial_score): 2")

df.describe().T[["min", "50%", "max"]].rename(columns={"50%": "median"})

# Assertions
assert len(df) == 21_925, f"Expected 21,925 rows, got {len(df)}"
assert len(df.columns) == 403, f"Expected 403 columns, got {len(df.columns)}"
missing = df.isna().sum().sum()
assert missing == 0, f"Found {missing} missing values"
assert df["quality_score"].between(0, 100).all(), "quality_score out of 0–100 range"
assert df["commercial_score"].between(0, 100).all(), (
    "commercial_score out of 0–100 range"
)
pandemic = df[df["BGGId"] == 30549].iloc[0]
assert pandemic["quality_score"] > 70, (
    f"Pandemic quality_score suspiciously low: {pandemic['quality_score']:.1f}"
)
assert pandemic["commercial_score"] > 70, (
    f"Pandemic commercial_score suspiciously low: {pandemic['commercial_score']:.1f}"
)
print(
    f"Pandemic — quality: {pandemic['quality_score']:.1f}, commercial: {pandemic['commercial_score']:.1f}"
)
print("All assertions passed.")

Missing values per column (should all be 0):
  None

Rows: 21,925
Cols: 403
  — continuous features: 8
  — binary (Cat + mechanics + themes + subcategories): 392
  — target labels (quality_score, commercial_score): 2
Pandemic — quality: 74.9, commercial: 100.0
All assertions passed.


## Save

In [53]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}  ({OUT_PATH.stat().st_size / 1e6:.1f} MB)")

Saved: ../data/games_processed.csv  (18.6 MB)
